# Tools

In [2]:
import os 
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content="Parrots aren’t really “talking” the way humans do—they’re masters of **vocal mimicry**. A few key factors make them especially good at copying human speech:\n\n| Factor | What it means for the bird | Why it matters |\n|--------|---------------------------|----------------|\n| **Highly developed syrinx** | The syrinx is the bird’s voice box, located at the base of the trachea. In parrots it’s exceptionally flexible, allowing them to produce a wide range of pitches, tones, and rapid modulations. | Gives them the raw acoustic capability to reproduce the nuances of human words. |\n| **Acute auditory memory** | Parrots can hear, store, and recall sounds with great precision, even after just a few repetitions. | Enables them to learn and retain new “words” after hearing them only a handful of times. |\n| **Social nature** | In the wild, parrots live in flocks where they constantly exchange calls to keep the group cohesive, signal danger, and maintain bonds. | Mimicking sou

In [5]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It is sunny in {location}."

model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks weather in New York. We can call function get_weather.', 'tool_calls': [{'id': 'fc_0cf952a2-2468-469d-bb33-6cf414d6913a', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 128, 'total_tokens': 171, 'completion_time': 0.090984228, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.005308268, 'prompt_tokens_details': None, 'queue_time': 0.212672783, 'total_time': 0.096292496}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c3da0d4bb9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a12bba-fc60-7d52-a4db-33cd803b7a78-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_0cf952a2-2468-469d-bb33-6cf414d6913a', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'i

### Tool Execution Loop

In [7]:
# step 1. model generates tool calls 
messages = [{"role": "user", "content": "What is the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)


# step 2. execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generate arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


# step 3. pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The current weather in New York is sunny.
